<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Assignment 2: RAG
  </span>
</h2>

<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Student Name: Yaron Winter
  </span>
</h2>

In [88]:
print("Install required packages:")
!pip install --upgrade pip

!pip install -q google
#!pip install -q google-colab
!pip install -q google-api-python-client
!pip install -q pypdf
!pip install -q langchain-text-splitters
!pip install -q faiss-cpu
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-openai
!pip install -q sentence-transformers
!pip install -q ragas
print("Done Install!")

Install required packages:
Done Install!


<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 1 - Naive Generation
  </span>
</h2>

In [2]:
import pandas as pd

TABLES_FOLDER = "/home/yaron/WorkEnv/data/nebius/weeks/week3/rag_and_ai/task/"
SORTED_TOP_10_TABLE = "top_10_questions.csv"
SORTED_FULL_TABLE = "sorted_by_id_benchmark.csv"

DISPLAY_FIELDS = ["financebench_id", "doc_name", "question_type", "question", "answer", "evidence"]

top_10_df = pd.read_csv(TABLES_FOLDER + SORTED_TOP_10_TABLE, index_col=0)
full_df = pd.read_csv(TABLES_FOLDER + SORTED_FULL_TABLE, index_col=0)

In [3]:
print("Tables (after removing metrics-generated + sorting by ID):")
print(f"len(top_10)={len(top_10_df)}, len(full)={len(full_df)}\n")
top_10_df[DISPLAY_FIELDS].head(5)

Tables (after removing metrics-generated + sorting by ID):
len(top_10)=10, len(full)=100



,financebench_id,doc_name,question_type,question,answer,evidence
0,financebench_id_00005,CORNING_2022_10K,domain-relevant,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,[{'evidence_text': 'Consolidated Balance Sheet...
1,financebench_id_00070,AMERICANWATERWORKS_2022_10K,domain-relevant,Does American Water Works have positive workin...,"No, American Water Works had negative working ...",[{'evidence_text': 'American Water Works Compa...
2,financebench_id_00080,PAYPAL_2022_10K,domain-relevant,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...,"[{'evidence_text': 'PayPal Holdings, Inc.\nCON..."
3,financebench_id_00206,JPMORGAN_2022_10K,domain-relevant,Are JPM's gross margins historically consisten...,"Since JPM is a financial institution, gross ma...",[{'evidence_text': 'Overview\nJPMorgan Chase &...
4,financebench_id_00215,VERIZON_2022_10K,domain-relevant,Is Verizon a capital intensive business based ...,Yes. Verizon's capital intensity ratio was app...,[{'evidence_text': 'Consolidated Balance Sheet...


In [4]:
# Get Nebius API Key
from getpass import getpass
#from google.colab import userdata
import os

def get_api_key(name="NEBIUS_API_KEY"):
    api_key = None
    try:
        api_key = os.environ[name]
        print("API Key is taken from the environmental parameters")
    except:
        try:
            api_key = userdata.get(name)
            print("API Key is taken from the google colab user data")
        except:
            try:
                api_key = getpass("Enter API key: ")
                print("API Key is taken from getpass")
            except:
                raise Exception("API Key for NEBIUS_API_KEY could not be found")

    assert api_key is not None, "API Key is None"
    return api_key

In [5]:
# Allocate AI client
from openai import OpenAI

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key = get_api_key(),
)

Enter API key:  ········


API Key is taken from getpass


In [6]:
# Define the LLM response generation code.
import json
def generate_response(attributes: dict,
                      model_name: str,
                      system_prompt: str,
                      user_prompt: str,
                      response_format: dict) -> dict:
    virtual_user_prompt = (user_prompt if attributes is None else f"{user_prompt} {json.dumps(attributes)}```")
    return client.chat.completions.create(
        model=model_name,
        response_format=response_format,
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": virtual_user_prompt}
                ],
            },
        ]
    ).to_dict()
print("Done!")

Done!


In [7]:
# Define the prompts for the naive answering generation.
NAIVE_SYSTEM_PROMPT = """
You are an AI agent.
Your task is to answer the questions given to you.
The question to be answered will be presented by a simple JSON-formatted string,
as given in this example:

***Example:***
"{
'Question':'Does Corning have positive working capital based on FY2022 data?...',
}"

***Output Format:***
Please retrive just the generated description, according to the provided format.
"""

NAIVE_USER_PROMPT = """
Please answer this given question.
The question is provided below as JSON-Formatted string between triple backticks:

```json
"""

NAIVE_RESPONSE_SCHEMA = {
  "name": "generated_answer",
  "schema": {
    "type": "object",
    "properties": {
      "generated_answer": {
        "type": "string",
        "description": "An answer to the given question. The length of the answer should not be longer than 90 words"
      },
    },
    "required":["generated_answer"],
    "additionalProperties": False
  },
  "strict": True
}

NAIVE_RESPONSE_FORMAT = {"type": "json_schema", "json_schema": NAIVE_RESPONSE_SCHEMA}

In [8]:
# The code for the naive answers generation.
QUESTION = "Question"

def generate_naive_answer(question: str, model_name: str) -> str:
    q = {QUESTION: question}
    response = generate_response(model_name=model_name,
                                 attributes=q,
                                 user_prompt=NAIVE_USER_PROMPT,
                                 system_prompt=NAIVE_SYSTEM_PROMPT,
                                 response_format=NAIVE_RESPONSE_FORMAT)
    j = json.loads(response["choices"][0]["message"]["content"])
    return j["generated_answer"]
print("Done!")

Done!


In [9]:
# Generate naive answers to the top 10 entries.
naive_df = top_10_df[["financebench_id", "question_type", "question"]].copy()
naive_df["ground_truth"] = top_10_df.answer
naive_df["naive_answer"] = naive_df.question.apply(lambda x: generate_naive_answer(x, model_name="meta-llama/Llama-3.3-70B-Instruct"))
naive_df["verdict"] = "DEFAULT"

In [10]:
# Save  the table for manual analysis.
naive_df.to_csv("assignment2_naive_generation.csv")

In [12]:
# Load the table following the manual analysis.
naive_df = pd.read_csv("assignment2_naive_generation.csv")
naive_df.head(10)

,Unnamed: 0,financebench_id,question_type,question,ground_truth,naive_answer,verdict
0,0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,Corning's FY2022 data shows total current asse...,wrong
1,1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,"No, American Water Works had negative working ...","According to FY2022 data, American Water Works...",partially correct
2,2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...,"According to FY2022 data, Paypal's current ass...",wrong
3,3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,"Since JPM is a financial institution, gross ma...","JPM's gross margins are not a relevant metric,...",correct
4,4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,Yes. Verizon's capital intensity ratio was app...,Verizon is considered a capital-intensive busi...,partially correct
5,5,financebench_id_00283,novel-generated,How much does Pfizer expect to pay to spin off...,77.78,Pfizer expects to pay approximately $12 billio...,wrong
6,6,financebench_id_00288,novel-generated,Was there any drop in Cash & Cash equivalents ...,"Yes, there was a decline of ~42% between FY202...",No data is provided to determine if there was ...,wrong
7,7,financebench_id_00299,novel-generated,Which of JPM's business segments had the lowes...,Corporate. Its net revenue was -$473 million.,Corporate segment had the lowest net revenue i...,correct
8,8,financebench_id_00302,novel-generated,Did Pfizer grow its PPNE between FY20 and FY21?,"Yes, change in PPNE was positive year over year","According to Pfizer's financial reports, its P...",partially correct
9,9,financebench_id_00382,novel-generated,Which region had the Highest EBITDAR Contribut...,Las Vegas resorts contributed ~90% of company ...,Las Vegas Strip had the highest EBITDAR contri...,correct


### **Analysis & Summary:**

    - There were no cases of refusal, perhaps due to the prompts and response format, which encouraged the LLM to generate an adequately formatted response.

    - The model was pretty confident in all its answers.

    - There were two cases in which the model answered very confidently on a numeric questions, but was completely wrong (lines 0 and 5 in the table); in another case (line 2) it generated response about the relevancy, which was wrong.

    - I could not recognize clear differences between the two question types. In fact, the distributions of the verdicts in both cases are very similar.

<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 2 - RAG Reminder
  </span>
</h2>


    - Indexing:
        - A storage of data relevant to the business / application, which can be crucial for generating adequate responses
        - The generation of this storage is done offline, thus spares resources in service time
        - The storage enables the system to remain updated - a convenient way to deal with dynamic or specific data
        - The text chunking might be tricky and induces failures, though
            - too short chunks may not include relevant context, thus may harm the response generation stage
            - too long chunks may reduce the semantic focus of the text, thus may harm the retrieval stage
            
    - Retrieval:
        - This stage is performed online, per query
        - It is used for bringing to the LLM the most relevant information, which is crucial for generating good response
        - It delivers the LLM only a small portion of the data, instead of it all, thus reduce costs and delay time
        - But failures in this stage induces generation failures
            - the LLM does not accept the information needed (relevant docs not retrieved)
            - the LLM accepts too much noise (many docs retrieved, for ensuring high retrieval recall)

    - Generation:
        - Performed online, of course, for each query
        - Does the essential task of the pipeline: generating a response
        - LLMs are particularly strong in analyzing given texts, so using the given docs may contribute a lot
        - But garbage in -> garbage out: the model depends heavily on the docs delivered to it

<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 3 - Embed Documents
  </span>
</h2>


In [13]:
# Load the documents and store them in a map, for future usage.
import glob
from pypdf import PdfReader

DOC_NAME = "doc_name"
COMPANY_NAME = "company_name"
DOC_PERIOD = "doc_period"
PAGE_NUMBER = "page_number"
TEXT = "text"

PDFS_FOLDER = "/home/yaron/WorkEnv/data/nebius/weeks/week3/rag_and_ai/financebench/pdfs/"

pdf_files = glob.glob(PDFS_FOLDER + "*.pdf")
doc_name_to_pdf = {os.path.splitext(os.path.basename(x))[0]:x for x in pdf_files}
print(f"type={type(doc_name_to_pdf)}, len={len(doc_name_to_pdf)}")
print(f"pdf docs:\n{list(doc_name_to_pdf.items())[:3]}")

type=<class 'dict'>, len=368
pdf docs:
[('VERIZON_2017_10K', '/home/yaron/WorkEnv/data/nebius/weeks/week3/rag_and_ai/financebench/pdfs/VERIZON_2017_10K.pdf'), ('WALMART_2018_10K', '/home/yaron/WorkEnv/data/nebius/weeks/week3/rag_and_ai/financebench/pdfs/WALMART_2018_10K.pdf'), ('ADOBE_2019_10K', '/home/yaron/WorkEnv/data/nebius/weeks/week3/rag_and_ai/financebench/pdfs/ADOBE_2019_10K.pdf')]


In [14]:
# Read and parse a pdf document.
def parse_pdf(attributes: dict, pdf_path: str) -> list:
    reader = PdfReader(pdf_path)
    return [
        {
            DOC_NAME: attributes[DOC_NAME],
            COMPANY_NAME: attributes[COMPANY_NAME],
            DOC_PERIOD: attributes[DOC_PERIOD],
            PAGE_NUMBER: i,
            TEXT: page.extract_text(),
        }
        for i, page in enumerate(reader.pages)
    ]

In [15]:
# Split a page into chunks.
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def split_page(page: dict,
               chunk_size=1000,
               chunk_overlap=150):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    base_metadata = {
        k: v for k, v in page.items() if k != TEXT
    }

    doc = Document(
        page_content=page[TEXT],
        metadata=base_metadata
    )

    return splitter.split_documents([doc])

In [16]:
# Embed the table's documents.
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from tqdm import tqdm

VECTOR_STORAGE_PATH = "/home/yaron/WorkEnv/data/nebius/weeks/week3/rag_and_ai/task/faiss_index"

embedder = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

def embed_table_docs(df: pd.DataFrame, chunk_size=1000, chunk_overlap=150):
    vectorstore = None
    
    handled_docs = set()

    for i in tqdm(range(len(df))):
        doc_name = df.loc[i, "doc_name"]
        if doc_name in handled_docs:
            continue

        handled_docs.add(doc_name)

        try:
            doc_period = df.loc[i, "doc_period"].item()
        except:
            doc_period = df.loc[i, "doc_period"]
            
        attributes = {DOC_NAME: doc_name, DOC_PERIOD: doc_period, COMPANY_NAME: df.loc[i,"company"]}
        pages = parse_pdf(attributes=attributes, pdf_path=doc_name_to_pdf[doc_name])

        total_doc_chunks = []
        for page in pages:
            total_doc_chunks.extend(split_page(page=page, chunk_size=chunk_size, chunk_overlap=chunk_overlap))

        if vectorstore is None:
            vectorstore = FAISS.from_documents(total_doc_chunks, embedder)
        else:
            vectorstore.add_documents(total_doc_chunks)

    vectorstore.save_local(f"{VECTOR_STORAGE_PATH}_cs_{chunk_size}_co_{chunk_overlap}")
print("Done")

/tmp/ipykernel_7900/387637001.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedder = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Done


In [96]:
# Perform the embedding.
embed_table_docs(df=full_df, chunk_size=1000, chunk_overlap=150)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [29:01<00:00, 17.41s/it]


In [17]:
# Perform some tests on the vector storage.
vectorstore = FAISS.load_local(
    "/home/yaron/WorkEnv/data/nebius/weeks/week3/rag_and_ai/task/faiss_index_cs_1000_co_150",
    embedder,
    allow_dangerous_deserialization=True
)

# Define a test /analysis function,for
# checking the retrieval performance
def display_retrieval_results(df: pd.DataFrame, ind: int, k: int, text_len: int):
    query = df.loc[ind, "question"]
    results = vectorstore.similarity_search(query, k=k)                                        
    print(f"Question: {query}:")
    print(f"\tcompany={df.loc[ind, "company"]}")
    print(f"\tdoc={df.loc[ind, "doc_name"]}")
    print(f"\tqtype={df.loc[ind, "question_type"]}")
    evidence = df.loc[ind,"evidence"]
    evidence = evidence.replace("\'", "\"")
    evidence = json.loads(evidence)[0]
    print(f"\tevidence: page={evidence["evidence_page_num"]}, text={evidence["evidence_text"][:text_len]}")
    print("\nRetrieved:")
    for r in results:
        print(r.metadata, r.page_content[:text_len])

In [18]:
display_retrieval_results(full_df, ind=15, k=3, text_len=50)

Question: Was there any change in the number of Best Buy stores between Q2 of FY2024 and FY2023?:
	company=Best Buy
	doc=BESTBUY_2024Q2_10Q
	qtype=novel-generated
	evidence: page=16, text=iscal 2024 was primarily driven by comparable sale

Retrieved:
{'doc_name': 'BESTBUY_2024Q2_10Q', 'company_name': 'Best Buy', 'doc_period': np.int64(2024), 'page_number': 16} Fiscal 2024  Fiscal 2023
 
Total Stores at Beginni
{'doc_name': 'BESTBUY_2023_10K', 'company_name': 'Best Buy', 'doc_period': np.int64(2023), 'page_number': 25} (1) Excludes stores that were temporarily closed a
{'doc_name': 'BESTBUY_2024Q2_10Q', 'company_name': 'Best Buy', 'doc_period': np.int64(2024), 'page_number': 18} Table of Contents International segment stores ope


### **Retrieval Performance Analysis:**

    - I have tested a few questions from the table, using the above function (display_retrieval_results)
    - In all cases, both the top and the majority of the retrieved chunks came from the correct doc
    - But the page number of the top retrieved chunk was correct only in ~40% of the cases
    - And when the correct page was not at top - it was never among the retrieved chunks, not even for k=15
    - In some cases the evidence text consisted on sequence of numbers
        - such lists of numbers cannot be similar to the textual question, of course

<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 4 - RAG Pipeline
  </span>
</h2>

In [105]:
# Define the prompts for the RAG pipeline.
RAG_SYSTEM_PROMPT = """
You are an AI agent who should answer questions based on a given context.
Your task is to analyze the context and find out how it can be used
for generating an adequate response to the question.
Notice that your answer must rely on the given context strictly and solely!

DO NOT USE OTHER KNOWLEDGE SOURCES FOR GENERAING AN ANSWER!

Your answers should be concise and be not longer than 100 words.
You should also cite the documents, based on which you generated the answer.
"""

def build_rag_user_prompt(question: str, k: int=4) -> tuple:
    """Build the RAG prompt with runtime data enhancement."""
    # Retrieve a list of chunks, which are most similar to the given question.
    results = vectorstore.similarity_search(query=question, k=k)

    if len(results) == 0:
        return "", ""

    contexts = []
    for doc in results:
        contexts.append(
            f"Doc Name: {doc.metadata[DOC_NAME]} | Text: {doc.page_content}"
        )

    context = "\n\n".join(contexts)

    return f"""Answer the given question based strictly on the provided context
If the context does not contain enough information, just say "No sufficient context is give".
Do not make up information.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:""", context


In [106]:
# Define the RAG pipeline

ANSWER = "answer"
RETRIEVED_DOCS = "retrieved_chunks"
CONTEXT = "context"

def answer_with_rag(question: str,
                    k: int=4,
                    model_name: str="meta-llama/Llama-3.3-70B-Instruct") -> dict:
    # Build the user prompt.
    user_prompt, context = build_rag_user_prompt(question=question, retrieved_docs=results)

    # Handle empty retrieved list.
    if len(context) == 0:
        return {ANSWER: "no retrieved docs", RETRIEVED_DOCS: []}

    # Generate the answer.
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=200
    )

    answer = response.choices[0].message.content.strip()
    return {ANSWER: answer, RETRIEVED_DOCS: results, CONTEXT: context}

<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 5 - Run & Compare
  </span>
</h2>


In [162]:
# Generate RAG based answers to the first 10 questions.
def get_rag_response(question: str,
                     k: int=4,
                     model_name: str="meta-llama/Llama-3.3-70B-Instruct") -> str:
    d = answer_with_rag(question=question, k=k, model_name=model_name)
    return d[ANSWER]
    
naive_df = pd.read_csv("assignment2_naive_generation.csv")

rag_df = naive_df.copy()
rag_df["RAG_answer"] = rag_df.question.apply(lambda x: get_rag_response(question=x))
rag_df["Comment"] = "DEFAULT"

In [21]:
#rag_df.to_csv("assignment2_run_and_compare.csv")

In [22]:
# After manual analysis
rag_df = pd.read_csv("assignment2_run_and_compare.csv", index_col=0)
rag_df.head(10)

,Unnamed: 0,financebench_id,question_type,question,ground_truth,naive_answer,verdict,RAG_answer,Comment
0,0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,Corning's FY2022 data shows total current asse...,wrong,No sufficient context is given.\n\nThe provide...,Cannot answer better than wrong?
1,1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,"No, American Water Works had negative working ...","According to FY2022 data, American Water Works...",partially correct,No sufficient context is given.\n\nThe provide...,naive was better
2,2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...,"According to FY2022 data, Paypal's current ass...",wrong,No sufficient context is given. The provided c...,Cannot answer better than wrong?
3,3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,"Since JPM is a financial institution, gross ma...","JPM's gross margins are not a relevant metric,...",correct,No sufficient context is given.\n\nThe provide...,naive was better (general knowledge helped)
4,4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,Yes. Verizon's capital intensity ratio was app...,Verizon is considered a capital-intensive busi...,partially correct,"Yes, Verizon is a capital-intensive business. ...",RAG was better! (correct numbers?)
5,5,financebench_id_00283,novel-generated,How much does Pfizer expect to pay to spin off...,77.78,Pfizer expects to pay approximately $12 billio...,wrong,No sufficient context is given.\n\nThe provide...,Cannot answer better than wrong?
6,6,financebench_id_00288,novel-generated,Was there any drop in Cash & Cash equivalents ...,"Yes, there was a decline of ~42% between FY202...",No data is provided to determine if there was ...,wrong,"Yes, there was a drop. Cash and cash equivalen...",RAG was better! (correct numbers?)
7,7,financebench_id_00299,novel-generated,Which of JPM's business segments had the lowes...,Corporate. Its net revenue was -$473 million.,Corporate segment had the lowest net revenue i...,correct,No sufficient context is given.\n\nThe provide...,naive was better:(
8,8,financebench_id_00302,novel-generated,Did Pfizer grow its PPNE between FY20 and FY21?,"Yes, change in PPNE was positive year over year","According to Pfizer's financial reports, its P...",partially correct,No sufficient context is given.\n\nThe provide...,naive was better:(
9,9,financebench_id_00382,novel-generated,Which region had the Highest EBITDAR Contribut...,Las Vegas resorts contributed ~90% of company ...,Las Vegas Strip had the highest EBITDAR contri...,correct,Las Vegas Strip Resorts had the highest EBITDA...,both were correct


### **Comparison Discussion:**

    - Overall and clearly: the RAG did not improve the performance, but rather the opposite.
    - In 8 of the 10 questions the RAG responded with "No sufficient context..."
        - in a way it is better than responding with a wrong answer
        - but when it is so frequent - there is a problem
        - need to check deeper, whether the relevant chunks are indeed retrieved
        - but as a lot of the data are long columns of numbers - there might be a generation problen even when relevant chunks are retrieved
    - There is a good point with the RAG, though:
        - in the cases for which it generated answers - the answers were correct!
    - I could not recognize different behavior of the RAG, across the two question types

<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 6 - Evaluation
  </span>
</h2>


In [169]:
# Generate answers for the whole dataset
rag_df = full_df[["financebench_id", "question_type", "question"]].copy()
rag_df["ground_truth"] = full_df.answer
rag_df["RAG_answer"] = rag_df.question.apply(lambda x: get_rag_response(question=x))
rag_df.head(5)

,financebench_id,question_type,question,ground_truth,RAG_answer
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,No sufficient context is given. \n\nThe provid...
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,"No, American Water Works had negative working ...",No sufficient context is given. \n\nThe provid...
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...,No sufficient context is given. The provided c...
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,"Since JPM is a financial institution, gross ma...",No sufficient context is given. The provided d...
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,Yes. Verizon's capital intensity ratio was app...,"Yes, Verizon is a capital-intensive business. ..."


In [84]:
rag_df.to_csv("assignment2_evaluation.csv")

In [24]:
rag_df = pd.read_csv("assignment2_evaluation.csv")

In [78]:
# Handle correctness
# Define the correctness judgement prompts
CORRECTNESS_SYS_PROMPT = """
You are an AI agent who provides a judging service.
Your task is to rate a given answer, which was generated
by another AI agent.
The generated answer is a response to a question.
Along with the generated answer you will be given also the ground truth
answer for that question.
The pair of answers will be represented by a JSON-formatted string,
consisting of two keys - ground_truth and generated_answer, as described
in the following example:

*** Query Example ***
"{
'ground_truth': 'Yes, there was a decline of ~42% between FY2023 and Q2 of FY 2024.',
'generated_answer': 'Yes, there was a drop. Cash and cash equivalents decreased from $1,874 in FY 2023 to $1,093 in Q2 of FY2024'
}"

Your task, therefore, is to compare between the two given answers, and rate
the generated answer as either correct or incorrect, based on the similarity
between its content to the content of the ground truth answer.
Here are some guidelines, according to which you should generate your verdict:

*** Example 1 ***:
Query:
{
'ground_truth': 'Yes, there was a decline of ~42% between FY2023 and Q2 of FY 2024.',
'generated_answer': 'Yes, there was a drop. Cash and cash equivalents decreased from $1,874 in FY 2023 to $1,093 in Q2 of FY2024'
}

Response:
{
verdict: correct
justification: the generated answer agrees that there is a decline, and also agree about the percentage of the decline
}

*** Example 2 ***:
Query:
{
'ground_truth': 'Las Vegas resorts contributed ~90% of company level EBITDAR during FY2022.',
'generated_answer': 'Las Vegas Strip Resorts had the highest EBITDAR contribution for MGM during FY2022, with $3.1 billion.'
}

Response:
{
verdict: correct
justification: the agent agrees that Las Vegas had the top EBITDAR contribution
}

*** Example 3 ***:
Query:
{
'ground_truth': 'No, American Water Works had negative working capital of -$1561M in FY 2022.',
'generated_answer': 'No sufficient context is given.\n\nThe provided context does not contain...'
}

Response:
{
verdict: incorrect
justification: the agent agrees admits that he could not generate an answer
}

In general, when the generated answer of the agent agrees with the ground truth answer
about the essense of the issue (e.g. decline or increase, answers to 'who' or 'which', etc.) - 
the verdict should be 'correct', even if the rest of the answers is not particularly similar.
If the essense of the answer is diffrent, or the agent admits that he could not generate
an answer - the verdict is 'incorrect'.

NOTICE: your verdict must be either 'correct' or 'incorrect'!!!
You must retrieve any other value as your verdict!

***Output Format:***
Please retrive a response according to the provided response_format structure.
Please provide for each criterion a justification, which explains your verdict choice.
"""

CORRECTNESS_USER_PROMPT = """
Please rate the given generated answer based on its similarity to the ground truth answer,
as explained above, and output your decision according to the provided structure.
The two answers are provided below as JSON-Formatted string between triple backticks:

```json
"""

In [79]:
VERDICT = "verdict"
JUSTIFY = "justification"
JUDGE = "Judgement"
CORRETNESS = "correctness"

# Define the response format
CORRECTNESS_RESPONSE_SCHEMA = {
  "name": "judge_angent",
  "schema": {
    "type": "object",
    "properties": {
      JUDGE: {
        "type": "object",
        "properties": {
          JUSTIFY: {
            "type": "string",
            "description": "explain your verdict in a single sentence"
          },
          VERDICT: {
            "type": "string",
            "description": "the correctness of the generated answer: correct / incorrect"
          },
        },
        "required": [
          JUSTIFY,
          VERDICT
        ],
        "additionalProperties": False
      },
    },
    "required":[JUDGE],
    "additionalProperties": False
  },
  "strict": True
}

CORRECTNESS_RESPONSE_FORMAT = {"type": "json_schema", "json_schema": CORRECTNESS_RESPONSE_SCHEMA}

In [80]:
# Judge the generated answers correctness.

GENERATED_ANSWER = "generated_answer"
GROUND_TRUTH = "ground_truth"

def rate_answer(r: pd.Series, model_name: str) -> tuple:
    attribute = {GENERATED_ANSWER: r["RAG_answer"], GROUND_TRUTH: r[GROUND_TRUTH]}
    response = generate_response(attribute,
                                 model_name,
                                 CORRECTNESS_SYS_PROMPT,
                                 CORRECTNESS_USER_PROMPT,
                                 response_format=CORRECTNESS_RESPONSE_FORMAT
                                )
    j = json.loads(response["choices"][0]["message"]["content"])
    return (j[JUDGE][VERDICT], j[JUDGE][JUSTIFY],)

In [81]:
# This is the only deepseek model I say in Nebius tokens factory.
# The model stated in the task (DeepSeek-V3-0324) was not there.
model_name = "deepseek-ai/DeepSeek-V3.2" 
judgements = rag_df.apply(lambda r: rate_answer(r, model_name=model_name), axis=1).tolist()
print("Done!")

Done!


In [83]:
verdicts = [x[0] for x in judgements]
justifications = [x[1] for x in judgements]
rag_df[CORRETNESS] = verdicts
rag_df[JUSTIFY] = justifications
rag_df.head(5)

,Unnamed: 0.1,Unnamed: 0,financebench_id,question_type,question,ground_truth,RAG_answer,correctness,justification
0,0,0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,No sufficient context is given.\n\nThe provide...,incorrect,The generated answer states there is insuffici...
1,1,1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,"No, American Water Works had negative working ...",No sufficient context is given.\n\nThe provide...,incorrect,The generated answer states that there is insu...
2,2,2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...,No sufficient context is given. The provided c...,incorrect,The agent's generated answer states that there...
3,3,3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,"Since JPM is a financial institution, gross ma...",No sufficient context is given. The provided d...,incorrect,The ground truth answer states that gross marg...
4,4,4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,Yes. Verizon's capital intensity ratio was app...,"Yes, Verizon is a capital-intensive business. ...",correct,The generated answer is being assessed on whet...


In [108]:
rag_df.to_csv("assignment2_evaluation.csv")

In [111]:
# Handle Faithfullness.
# Notice that we were instructed to use AsyncOpenAI,
# while using the syncronious score computation.
# This cause an error, with no elegant way to resolve it.
# After some digging I found out, that the two optimal
# ways to handle this issue are either making it all
# async, or using evaluate component, for keeping
# things sync.
# I chose, this, tokeep everything sync by using evaluate.

from ragas.metrics import faithfulness
from ragas import evaluate
from langchain_openai import ChatOpenAI
from datasets import Dataset

# Define LLM (SYNC)
model_name = "deepseek-ai/DeepSeek-V3.2" 
llm = ChatOpenAI(
    model=model_name,
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=get_api_key(),
)

/tmp/ipykernel_7900/2581393556.py:11: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness


Enter API key:  ········


API Key is taken from getpass


In [127]:
# Compute faiyjfullness.
from datasets import Dataset
from ragas import evaluate

# Prepare dataset
sub_rag_df = rag_df[:20].copy()
questions = sub_rag_df["question"].tolist()
answers = sub_rag_df["RAG_answer"].tolist()
contexts = [[build_rag_user_prompt(question=q, k=4)[1]] for q in questions]

dataset = Dataset.from_dict({
        "question": questions,
        "answer": answers,
        "contexts": contexts
    })

results = evaluate(dataset, metrics=[faithfulness], llm=llm)
res_df = results.to_pandas()

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

In [133]:
faithfullness_res = res_df.faithfulness.tolist() + [-1.0]*(len(rag_df) - 20)
rag_df["faithfulness"] = faithfullness_res
rag_df.to_csv("assignment2_evaluation.csv")

In [146]:
# Handle retrieval hit rate.
def retrieval_hit(df: pd.DataFrame, ind: int, k: int) -> int:
    try:
        query = df.loc[ind, "question"]
        evidence = df.loc[ind,"evidence"]
        evidence = evidence.replace("\'", "\"")
        evidence = json.loads(evidence)[0]
        evidence_page_num = evidence["evidence_page_num"]
    except Exception as e:
        #print(f"Failed on index {i}, e={e}")
        return 0

    results = vectorstore.similarity_search(query, k=k)                                        
    for r in results:
        if r.metadata['page_number'] == evidence_page_num:
            return 1
    return 0

In [147]:
# Compute retrieval hits.
for k in {1,3,5}:
    hits = []
    for i in range(len(full_df)):
        hits.append(retrieval_hit(df=full_df, ind=i, k=k))
    rag_df[f"page_hit_at_{k}"] = hits

In [153]:
rag_df.to_csv("assignment2_evaluation.csv")

In [158]:
# Get average performance:
print(f"Mean Correctness\t{rag_df["correctness"].apply(lambda x: 1 if x=="correct" else 0).mean():.3f}")
print(f"Mean Faithfulness\t{rag_df[:20]["faithfulness"].mean():.3f}")
print(f"Mean hit@1\t{rag_df["page_hit_at_1"].mean():.3f}")
print(f"Mean hit@3\t{rag_df["page_hit_at_3"].mean():.3f}")
print(f"Mean hit@5\t{rag_df["page_hit_at_5"].mean():.3f}")

Mean Correctness	0.320
Mean Faithfulness	0.743
Mean hit@1	0.170
Mean hit@3	0.260
Mean hit@5	0.310


<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 7 - Improvement Cycles
  </span>
</h2>
